# nvs3d — Colab Training Runner

Train NeRF / 3DGS from scratch on the Lego scene using a Colab GPU.

**Prerequisites:** Upload `nerf_synthetic.zip` (or just the `lego/` folder) to your Google Drive.

See `docs/COLAB.md` in the repo for full instructions.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone the repo

In [ ]:
# TODO: REPLACE with your repo URL
REPO_URL = "https://github.com/YOUR_USER/nvs3d.git"  # TODO: REPLACE

!git clone $REPO_URL /content/nvs3d
%cd /content/nvs3d

## 3. Install dependencies

In [ ]:
!bash scripts/colab_setup.sh

## 4. Link Lego data from Drive

Adjust the source path to wherever you stored the lego folder on Drive.

In [ ]:
from pathlib import Path

# TODO: REPLACE with the path to lego/ on your Drive
DRIVE_LEGO = Path("/content/drive/MyDrive/datasets/nerf_synthetic/lego")  # TODO: REPLACE

LOCAL_LEGO = Path("/content/nvs3d/datasets/nerf_synthetic/lego")
LOCAL_LEGO.parent.mkdir(parents=True, exist_ok=True)

if not LOCAL_LEGO.exists():
    LOCAL_LEGO.symlink_to(DRIVE_LEGO)
    print(f"Linked {LOCAL_LEGO} -> {DRIVE_LEGO}")
else:
    print(f"Data already at {LOCAL_LEGO}")

# Verify
!python scripts/download_lego.py --verify --dest {LOCAL_LEGO}

## 5. Train

Output goes to Google Drive so a disconnect won't lose the run.

In [ ]:
# TODO: REPLACE config and flags as needed for the current phase
# Example: NeRF on Lego (P3+)

DRIVE_RUNS = "/content/drive/MyDrive/nvs3d_runs"  # TODO: REPLACE if desired

!python scripts/train.py \
    --config configs/nerf_blender.yaml \
    --data_dir {LOCAL_LEGO} \
    --output_dir {DRIVE_RUNS} \
    --device cuda

## 6. Resume from checkpoint

If the session disconnected, re-run cells 1–4 above, then this cell.

In [ ]:
# TODO: REPLACE with the actual run directory and checkpoint path
# The checkpoint loading logic uses the P0 checkpoint module.

from pathlib import Path

DRIVE_RUNS = Path("/content/drive/MyDrive/nvs3d_runs")  # TODO: REPLACE

# Find the latest run directory
run_dirs = sorted(DRIVE_RUNS.glob("*"), key=lambda p: p.name)
if run_dirs:
    latest_run = run_dirs[-1]
    ckpts = sorted((latest_run / "ckpt").glob("*.pt"))
    if ckpts:
        latest_ckpt = ckpts[-1]
        print(f"Latest checkpoint: {latest_ckpt}")
        # TODO: REPLACE — actual resume command depends on the phase
        # Example:
        # !python scripts/train.py --config configs/nerf_blender.yaml \
        #     --resume {latest_ckpt} --device cuda \
        #     --output_dir {DRIVE_RUNS}
    else:
        print(f"No checkpoints found in {latest_run / 'ckpt'}")
else:
    print(f"No runs found in {DRIVE_RUNS}")